In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

print(metadata_df.shape)
metadata_df


(17, 5)


,sample_title,tumor type,"timing of immunotherapy use relative to rt (1 - after only, 2 - before and after)",status (alive - 0; dead/hospice - 1),months to death or last fu
samples,,,,,
GSM3785160,"SM-HCP (YY, YZ, Z1)",Metastatic Melanoma,1,0,22.5666666666667
GSM3785161,SM-HCP (ZA-ZD),Metastatic Melanoma,2,1,2.8
GSM3785162,SM-HCQ6 (S-V),Metastatic Melanoma,1,1,24.2333333333333
GSM3785163,SM-HCP (WL-WP),Metastatic Melanoma,2,1,16.0333333333333
GSM3785164,SM-HCQ7 (D-L),Metastatic Melanoma,1,0,51.9666666666667
GSM3785165,"SM-HCP (VZ, W1-W7)",Metastatic Melanoma,1,1,25.0333333333333
GSM3785166,SM-HCP (WC-WF),Metastatic Melanoma,1,0,53.1
GSM3785167,SM-HCP (XP-XT),Metastatic Melanoma,2,1,23.1333333333333
GSM3785168,SM-HCP (Y4-Y8),Metastatic Melanoma,1,0,37


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_92442/3050319327.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tumor type,"timing of immunotherapy use relative to rt (1 - after only, 2 - before and after)",status (alive - 0; dead/hospice - 1),months to death or last fu,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM3785160,"SM-HCP (YY, YZ, Z1)",Metastatic Melanoma,1,0,22.5666666666667,20,3931,21,0,4,...,876,2,2,0,715,195,30,524,1,7
GSM3785161,SM-HCP (ZA-ZD),Metastatic Melanoma,2,1,2.8,64,3262,20,1,7,...,1073,2,1,1,895,300,27,416,1,10
GSM3785162,SM-HCQ6 (S-V),Metastatic Melanoma,1,1,24.2333333333333,159,8718,57,0,3,...,694,1,0,3,375,89,17,329,0,4
GSM3785163,SM-HCP (WL-WP),Metastatic Melanoma,2,1,16.0333333333333,48,4158,4,0,35,...,1691,2,29,29,1341,462,61,721,1,19
GSM3785164,SM-HCQ7 (D-L),Metastatic Melanoma,1,0,51.9666666666667,142,6265,58,0,6,...,697,13,1,2,613,195,33,481,0,3
GSM3785165,"SM-HCP (VZ, W1-W7)",Metastatic Melanoma,1,1,25.0333333333333,10,2375,0,0,0,...,743,1,0,0,531,109,12,215,12,2
GSM3785166,SM-HCP (WC-WF),Metastatic Melanoma,1,0,53.1,26,6553,43,0,3,...,2624,39,4,4,2386,840,161,1040,6,21
GSM3785167,SM-HCP (XP-XT),Metastatic Melanoma,2,1,23.1333333333333,15,4512,3,0,3,...,142,2,2,0,187,20,1,140,0,0
GSM3785168,SM-HCP (Y4-Y8),Metastatic Melanoma,1,0,37,90,5239,47,0,9,...,1970,1,1,2,1281,430,35,943,1,3


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
